In [16]:

# text = "I am Muhammad Ahtisham, student of bachelors science pursuing Artificial Intelligence from Aror University sukkur. I am top 10 GitHub commiter"
with open("/content/TwitterConvCorpus.txt", "r") as file:
    text = str(file.read())

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoConfig

def perform_ner(text, model_name="dbmdz/bert-large-cased-finetuned-conll03-english"):
    try:
        # Load model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(model_name)
        config = AutoConfig.from_pretrained(model_name)

        # Tokenize input
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

        # Get predictions
        with torch.no_grad():
            outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)

        # Get tokens and their predictions
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        token_labels = [config.id2label[p.item()] for p in predictions[0]]

        # Process results, handling special tokens and subwords
        results = []
        current_entity = []
        current_label = None

        # Skip [CLS] and [SEP]
        for token, label in zip(tokens[1:-1], token_labels[1:-1]):
            # Handle subwords
            if token.startswith("##"):
                if current_entity:
                    current_entity[-1] += token[2:]
                continue

            # Handle entity continuation
            if label.startswith("B-") or label == "O":
                if current_entity:
                    results.append((" ".join(current_entity), current_label))
                    current_entity = []
                # Remove B- prefix
                if label != "O":
                    current_entity = [token]
                    current_label = label[2:]
            elif label.startswith("I-"):
                if not current_entity:
                    current_entity = [token]
                    current_label = label[2:]
                else:
                    current_entity.append(token)

        # Add final entity if exists
        if current_entity:
            results.append((" ".join(current_entity), current_label))

        return results

    except Exception as e:
        print(f"Error performing NER: {str(e)}")
        return []



results = perform_ner(text)
print(results)

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[('Muhammad Ahtisham', 'PER'), ('Intelligence', 'MISC'), ('Aror University', 'ORG'), ('GitHub', 'MISC')]


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoConfig

def perform_ner(text, model_name="dbmdz/bert-large-cased-finetuned-conll03-english"):
    try:
        # Load model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(model_name)
        config = AutoConfig.from_pretrained(model_name)

        # Tokenize input
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

        # Get predictions
        with torch.no_grad():
            outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)

        # Get tokens and their predictions
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        token_labels = [config.id2label[p.item()] for p in predictions[0]]

        # Process results, handling special tokens and subwords
        results = []
        current_entity = []
        current_label = None

        # Skip [CLS] and [SEP]
        for token, label in zip(tokens[1:-1], token_labels[1:-1]):
            # Handle subwords
            if token.startswith("##"):
                if current_entity:
                    current_entity[-1] += token[2:]
                continue

            # Handle entity continuation
            if label.startswith("B-") or label == "O":
                if current_entity:
                    results.append((" ".join(current_entity), current_label))
                    current_entity = []
                # Remove B- prefix
                if label != "O":
                    current_entity = [token]
                    current_label = label[2:]
            elif label.startswith("I-"):
                if not current_entity:
                    current_entity = [token]
                    current_label = label[2:]
                else:
                    current_entity.append(token)

        # Add final entity if exists
        if current_entity:
            results.append((" ".join(current_entity), current_label))

        return results

    except Exception as e:
        print(f"Error performing NER: {str(e)}")
        return []

def filter_ner_results_by_label(all_results, target_label):
    """Filters the list of (entity, label) tuples to include only the target_label."""
    # Ensure the target label is upper-cased for matching (e.g., 'loc' becomes 'LOC')
    target_label = target_label.upper()

    filtered_entities_with_label = [
        (entity, label) for entity, label in all_results if label == target_label
    ]
    return filtered_entities_with_label



target_label = "MISC"

all_results = perform_ner(text)

filtered_results = filter_ner_results_by_label(all_results, target_label)

print(f"--- Results for Entity Type: {target_label} ---")
print(filtered_results)

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


--- Results for Entity Type: MISC ---
[('Cup', 'MISC')]
